# Correct A Reconstruction

Correction fixes artifacts that are already present in a reconstruction. In this tutorial, you will find very small diameters, correct a z-jump, and correct shrinkage artifacts.

## Workflow

```text
sample SWC files
      |
      v
find a suspicious diameter
      |
      v
view and correct the node
      |
      v
find and correct a z-jump
      |
      v
inspect and correct shrinkage
```

## Reading the correction commands

Correction uses `swc find` to locate suspicious nodes and `swc repair` to write corrected files.

| Part | Meaning |
| --- | --- |
| `swc find` | find nodes that match a condition |
| `-d 0.1 --comp lt` | find diameters smaller than `0.1` um |
| `-z 10` | find z-jumps larger than `10` um |
| `swc repair -d node-ids` | correct diameters at selected nodes |
| `swc repair -z node-ids --zjump join` | correct selected z-jumps |
| `-k 1.25 -kxy 1.1` | correct shrinkage in z and xy |
| `-o output/file.swc` | save the corrected file |

## Command helper

Run this cell once before the tutorial commands. It creates `shell_cmd()`, a small notebook helper that runs terminal commands, shows their output, and displays images when a command creates one.

In [ ]:
import shutil
import subprocess
from pathlib import Path

from IPython.display import Image, display


def shell_cmd(command, image=None):
    print("Executed command:")
    print(command)
    bash_path = shutil.which("bash")
    result = subprocess.run(
        command,
        shell=True,
        executable=bash_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    output = result.stdout.rstrip()
    print("
Output:")
    print(output if output else "(empty)")

    if image is not None and Path(image).exists():
        display(Image(filename=image))


## Prepare working folders

Create temporary folders for input data and generated output files.

In [ ]:
shell_cmd("mkdir -pv data output")

## Copy sample files

Use a simple branch for diameter correction, a small file with a z-jump, and an NMO reconstruction for shrinkage correction.

In [ ]:
shell_cmd("cp -v ../tests/data/pass_simple_branch.swc data/")
shell_cmd("cp -v ../tests/data/pass_zjump.swc data/")
shell_cmd("cp -v ../tests/data/pass_nmo_3_cut.swc data/")

## Find a small diameter

Find basal dendrite nodes with diameter smaller than `0.1` um.

In [ ]:
shell_cmd("f=data/pass_simple_branch.swc; smalld=$(swc find $f -p 3 -d 0.1 --comp lt); echo $smalld")

## Check where it is

Mark the node before correcting it.

In [ ]:
shell_cmd(
    "f=data/pass_simple_branch.swc; smalld=$(swc find $f -p 3 -d 0.1 --comp lt); swc view $f -j xy -m $smalld --show-id --no-axes -o output/small_diameter.png",
    image="output/small_diameter.png",
)

## Correct the diameter

Use the same node ID to correct the diameter and save the result.

In [ ]:
shell_cmd("f=data/pass_simple_branch.swc; smalld=$(swc find $f -p 3 -d 0.1 --comp lt); swc repair $f -d $smalld -o output/fixdiam.swc")
shell_cmd("cat output/fixdiam.swc")

The radius of node `5` should now be similar to its neighbors, nodes `4` and `6`.

## Find a z-jump

A z-jump is a sudden jump along the z-axis. This example finds a jump and marks it before correction.

In [ ]:
shell_cmd("f=data/pass_zjump.swc; zjumps=$(swc find $f -z 10); echo $zjumps")
shell_cmd(
    "f=data/pass_zjump.swc; zjumps=$(swc find $f -z 10); swc view $f -j xz -m $zjumps --show-id --no-axes -o output/zjump_points.png",
    image="output/zjump_points.png",
)

## Correct the z-jump

This example uses `--zjump join`. Other z-jump options are described in `docs/`.

In [ ]:
shell_cmd("f=data/pass_zjump.swc; zjumps=$(swc find $f -z 10); swc repair $f -z $zjumps --zjump join -o output/fixzjump.swc")
shell_cmd(
    "swc view output/fixzjump.swc data/pass_zjump.swc -j xz -c shadow --no-axes -o output/zjump_corrected.png",
    image="output/zjump_corrected.png",
)

## Inspect shrinkage

Start by viewing the reconstruction from the side.

In [ ]:
shell_cmd(
    "f=data/pass_nmo_3_cut.swc; swc view $f -j xz -o output/shrink_before.png",
    image="output/shrink_before.png",
)

## Correct shrinkage

Apply correction factor `1.25` in the z-axis and `1.1` in x-y.

In [ ]:
shell_cmd("f=data/pass_nmo_3_cut.swc; swc repair $f -k 1.25 -kxy 1.1 -o output/fixshrink.swc")
shell_cmd(
    "swc view output/fixshrink.swc data/pass_nmo_3_cut.swc -j xz -c shadow -o output/shrink_compare.png",
    image="output/shrink_compare.png",
)